# 01 — Data Pipeline (Data Engineer owns)

Prepares every data artifact the agent needs **before** it can answer a question. Run top to bottom. Imports the building blocks from `src/nrcs_navigator/data`.

**Persistence:** one PostgreSQL database with the pgvector extension holds the `payment_rates` table and the eCFR embeddings (and later the agent checkpointer).

**Sources**
1. NRCS Practice FIPS CSV (FY2023–FY2025) → `payment_rates` table → powers `payment_estimator`
2. Four eCFR parts (1466, 1468, 1470, 1464) fetched from the eCFR API as XML → chunked by section → embedded into the pgvector store for `eligibility_screener`

The two live scrape sources (practice standards, ranking dates) are fetched at query time, not here.

## Setup

Load config (incl. DATABASE_URL) and initialize the database: create the pgvector extension and the payment_rates schema.

In [ ]:

from nrcs_navigator.data import db

# Create the pgvector extension and the payment_rates table. Idempotent:
# CREATE ... IF NOT EXISTS means re-running the notebook is always safe.
db.init_db()
print("schema ready")

schema ready


## 1. Load the NRCS Practice FIPS CSV into payment_rates

In [2]:
from nrcs_navigator.data import fips_payments

# Read the raw FIPS CSV, normalize columns/types, and filter to in-scope
# programs (state-level rows, FY2023-2025), then write the cleaned rows into
# the Postgres payment_rates table. write() truncates first, so re-running the
# notebook reloads cleanly instead of stacking duplicate rows.
payments = fips_payments.load_clean()
rows = fips_payments.write(payments)
print(f"{rows:,} rows written to payment_rates")
payments.head()

19,285 rows written to payment_rates


,state,program,practice_code,practice_name,fiscal_year,instance_count,dollars_obligated,avg_payment_per_instance
0,Alabama,CSP-GCI,E300GCI,Grassland Conservation Initiative,2023,95,52961,557.48
1,Alabama,CSP-GCI,E300GCI,Grassland Conservation Initiative,2024,70,66915,955.93
2,Alabama,CStwP Farm Bill,E449C,"Advanced Automated IWM - Year 2-5, soil moistu...",2025,5,12160,2432.00
3,Alabama,CStwP Farm Bill,314,Brush Management,2023,28,11584,413.71
4,Alabama,CStwP Farm Bill,314,Brush Management,2025,16,272,17.00


## 2. Fetch and chunk the eCFR regulations (API → sections)

In [3]:
from nrcs_navigator.data import ecfr_loader

# Fetch the four in-scope eCFR parts from the API as structured XML (cached to
# data/raw), then chunk by section: one chunk per section, carrying its program,
# CFR part, section number, and citation. A section longer than CHUNK_SIZE
# tokens is split further, each piece keeping the section metadata.
chunks = ecfr_loader.load_chunks()
sections = {(c.metadata["part"], c.metadata["section"]) for c in chunks}
print(f"{len(chunks)} chunks from {len(sections)} sections")
chunks[0].metadata

208 chunks from 115 sections


{'program': 'EQIP',
 'part': '1466',
 'section': '1466.1',
 'citation': '7 CFR 1466.1',
 'heading': '§ 1466.1 Applicability.'}

## 3. Embed chunks into the pgvector store

In [4]:
from nrcs_navigator.data import vectorstore

# Embed the chunks with text-embedding-3-small and load them into the
# 'ecfr_regulations' pgvector collection. Rebuilds from scratch (pre_delete) so
# re-running replaces the vectors instead of duplicating them. Needs OPENAI_API_KEY.
n = vectorstore.build_index(chunks)
print(f"embedded and stored {n} chunks")

embedded and stored 208 chunks


## 4. Smoke check

Confirm payment_rates queries return rows and the pgvector store returns matches.

In [5]:
import pandas as pd

from nrcs_navigator.data import db, vectorstore

engine = db.get_engine()

# (a) Payments landed: total rows and distinct in-scope programs.
print(pd.read_sql(
    "SELECT count(*) AS rows, count(DISTINCT program) AS programs FROM payment_rates",
    engine,
))

# (b) eCFR store answers semantically, returning sections with citations --
# the shape eligibility_screener depends on.
print("\nsimilarity search: 'Am I eligible for EQIP if I rent my farmland?'")
for h in vectorstore.similarity_search("Am I eligible for EQIP if I rent my farmland?", k=3):
    print(f"  [{h.metadata['citation']}] {h.metadata['heading']}")

# (c) Sample of the lookup payment_estimator will run (filter by program + state).
pd.read_sql(
    """
    SELECT state, program, practice_code, practice_name, fiscal_year, avg_payment_per_instance
    FROM payment_rates
    WHERE program = 'EQIP Farm Bill' AND state = 'Alabama'
    ORDER BY fiscal_year, practice_code
    LIMIT 5
    """,
    engine,
)

    rows  programs
0  19285         7

similarity search: 'Am I eligible for EQIP if I rent my farmland?'


  [7 CFR 1466.6] § 1466.6 Program requirements.
  [7 CFR 1466.20] § 1466.20 Application for contracts and selecting applications.
  [7 CFR 1466.1] § 1466.1 Applicability.


,state,program,practice_code,practice_name,fiscal_year,avg_payment_per_instance
0,Alabama,EQIP Farm Bill,102,Comprehensive Nutrient Management Plan,2023,5854.18
1,Alabama,EQIP Farm Bill,106,Forest Management Plan,2023,3949.86
2,Alabama,EQIP Farm Bill,138,Conservation Plan Supporting Organic Transition,2023,5458.25
3,Alabama,EQIP Farm Bill,228,Agricultural Energy Assessment,2023,3192.84
4,Alabama,EQIP Farm Bill,313,Waste Storage Facility,2023,14251.78
